# CARBAMYLATION ANALYSIS TOOL

Import packages required for analysis of logfiles

In [1]:
import os

#data handling
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

#plotting
import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox
#import dash

#carbamylation analysis procedures
import postprocessing as postproc

Provide a logfile and a folder containing the postprocessed PDB and FASTA sequences.

In [2]:
fname = "results\\results_2608\\results.csv" #logfile
folder_path = "test" #folder containing PDB and fasta

Load logfile and plot its whole content

In [3]:
df = pd.read_csv(fname)
#origin = np.array([a.split("-")[0] for a in df["PDB Code"].values])
#df["origin"] = origin
df_aggregate = postproc.average_prot(df)

> Averaging Uniprot data...
>> P0CG48
>> Q02539
>> Q42589
>> P02925
>> P37802
>> P55265
>> P16401
>> P60709
>> P05020
>> P39207
>> Q9M2U7
>> O60506
>> O75531
>> Q969X6
>> P09211
>> P22102
>> Q8N257
>> Q9ULV4
>> Q14568
>> Q16531
>> P15311
>> Q9NW13
>> P0DMV8
>> P49005
>> P61224
>> P63267
>> P11498
>> P35580
>> Q6PEY2
>> Q96PC5
>> Q70AC7
>> Q12906
>> Q05639
>> Q562R1
>> P10724
>> P06733
>> Q58FF7
>> Q9Y285
>> Q9BXP5
>> P0ACF8
>> P00338
>> P29966
>> P31946
>> Q9UIJ7
>> Q8N142
>> P0A6F5
>> Q9HB71
>> P12277
>> Q12931
>> P11021
>> O00567
>> P36683
>> P48668
>> Q8N3F8
>> P22188
>> P54886
>> P68032
>> P16402
>> P0AEQ3
>> Q6FI81
>> Q8NBJ5
>> P39023
>> P0A9P0
>> P04406
>> Q99832
>> P17987
>> P18669
>> P06748
>> O76021
>> Q9XFT3
>> P63261
>> Q00796
>> P0A6B4
>> P14489
>> P16403
>> P15924
>> P62258
>> P60981
>> P67809
>> Q63ZY6
>> P0A7K2
>> O00471
>> P39377
>> P36578
>> P0DMV9
>> P10412
>> Q9SMU8
>> P61981
>> P10809
>> Q41932
>> P62736
>> P62750
>> P63104
>> Q9Y2R4
>> P34931
>> P13861
>> P31930
>>

Plot data on interactive plot. Gray points represent the aggregated data, featuring one point per uniprot code. Clicking on any point will show additional points in red. These points are the measures performed on every protein structure associated with the uniprot code selected by clicking.

In [21]:
#ISSUE: hovering always looks as if the aggregate plot is selected, although focus items are clicked

df_data = df

def update_point(trace, points, selector):
          
    #print(trace, points)    
    
    # only work with gray points
    if len(points.point_inds) == 0:
        return  
    
    # gather uniprot and chain ID from full dataset clicked point
    idx = points.point_inds[0]
    my_uniprot = df_data.loc[idx, "Uniprot Entry"]
    my_resid = df_data.loc[idx, "resid"]
    my_label = "%s(%i)"%(my_uniprot, my_resid)
    df_query = df.query('`Uniprot Entry` == "%s" and resid == %s'%(my_uniprot, my_resid))

    #plot data associated with selection
    #labels = ["UNIPROT: %s<br>resid: %i"%(df_query["Uniprot Entry"].values[i], df_query["resid"].values[i]) for i in range(len(df_query))]
    labels = ["PDB: %s"%(df_query["PDB Code"].values[i]) for i in range(len(df_query))]
    if len(f.data)>1:
        f.data = [f.data[0]]

    f.add_scatter(x=df_query["sasa"], y=df_query["pKa"],
                  mode='markers', showlegend=False, name=my_label,
                  text = labels, hovertemplate='%{text}<br>SASA: %{x:.2f}<br>pKa: %{y:.2f}')

    
    #identify clicked point and mark it differently (e.g. different colour or size)
    #c = list(scatter.marker.color)
    #for i in points.point_inds:
    #    c[i] = '#ffffff'
    #    with f.batch_update():
    #        scatter.marker.color = c

# create scatter plot
labels = ["UNIPROT: %s<br>resid: %i"%(df_data["Uniprot Entry"].values[i], df_data ["resid"].values[i]) for i in range(len(df_data))]
f = go.FigureWidget([go.Scatter(x=df_data["sasa"], y=df_data["pKa"],
                                mode='markers', name="aggregate", showlegend=False, opacity=0.75,
                                text = labels, hovertemplate='%{text}<br>SASA: %{x:.2f}<br>pKa: %{y:.2f}'
                               )
                    ])

# add labels
f.update_layout(
    xaxis_title="SASA (A2)",
    yaxis_title="pKa")

#set axes properties
f.update_xaxes(range=[0, 130])
f.update_yaxes(range=[-2, 20])
f.update_xaxes(showspikes=True)
f.update_yaxes(showspikes=True)

#parameterize colours and triggers
scatter = f.data[0]
colors = ['#aaaaaa'] * len(df_data)
scatter.marker.color = colors
f.layout.hovermode = 'closest'
scatter.on_click(update_point)
f


FigureWidget({
    'data': [{'hovertemplate': '%{text}<br>SASA: %{x:.2f}<br>pKa: %{y:.2f}',
              'mar…

Select desired uniprot entry and resid from the logfile, based on observation of the plot above.

In [16]:
uniprot = "P15531" #insert uniprot code here
resid = 85 #insert resid here

Display all lysine entries matching the uniprot code and resid provided above, and provide statistics(this is useful if more than one lysine conformation was analysed).

In [17]:
df_query = df.query('`Uniprot Entry` == "%s" and resid == %s'%(uniprot, resid))
pKa = df_query["pKa"]
sasa = df_query["sasa"]
print("uniprot %s, resid %s\n%s measures\npKa %4.2f pm %4.2f [%4.2f %4.2f]\nsasa %4.2f pm %4.2f [%4.2f %4.2f] A2"%
      (uniprot, resid, len(pKa),
       np.mean(pKa), np.std(pKa), np.min(pKa), np.max(pKa),
       np.mean(sasa), np.std(sasa), np.min(sasa), np.max(sasa)))


df.query('`Uniprot Entry` == "%s" and resid == %s'%(uniprot, resid))

uniprot P15531, resid 85
9 measures
pKa 8.98 pm 2.15 [4.50 10.47]
sasa 79.40 pm 12.21 [59.78 94.37] A2


,Unnamed: 0,resid,chain,pKa,sasa,PDB Code,Chain_Resid,Assembled Index,plddt,Uniprot Entry
8111,4,85,A,4.50,59.782414,1UCN-alt-1_assembled,A85,675.0,0.0,P15531
8116,9,85,B,6.19,66.484478,1UCN-alt-1_assembled,B85,1870.0,0.0,P15531
8135,28,85,F,7.76,78.682235,1UCN-alt-1_assembled,F85,6650.0,0.0,P15531
8144,7,85,A,10.47,82.435391,2HVD-alt-1_assembled,A85,672.0,0.0,P15531
8154,17,85,B,10.47,94.365066,2HVD-alt-1_assembled,B85,1870.0,0.0,P15531
8164,27,85,C,10.07,63.937694,2HVD-alt-1_assembled,C85,3068.0,0.0,P15531
14097,7,85,A,10.47,89.673621,3L7U-alt-1_assembled,A85,680.0,0.0,P15531
14107,17,85,B,10.44,88.199167,3L7U-alt-1_assembled,B85,1902.0,0.0,P15531
14117,27,85,C,10.44,91.014034,3L7U-alt-1_assembled,C85,3108.0,0.0,P15531


Select index of an entry of interest, to be visualized in 3D

In [18]:
idx = 8699

Display the structure of interest

In [19]:
resid = df.loc[idx, "resid"]
chain = df.loc[idx, "chain"]
PDB = "%s%s%s.pdb"%(folder_path, os.sep, df.loc[idx, "PDB Code"])
print("file: %s, chain: %s, resid: %s"%(PDB, chain, resid))

#u = mda.Universe(PDB)
#protein = u.select_atoms('protein')
#view = nv.show_mdanalysis(protein)
view

file: test\1E3U-alt1B_assembled.pdb, chain: D, resid: 205


NameError: name 'mda' is not defined

In [26]:
# SCATCH AREA, EXAMPLE CODES FOR DIFFERENT USAGES #

# example: show data visible through hovering over points
#
#fig = px.scatter(df_datae, x="sasa", y="pKa", opacity=0.5,
#                 #error_x="sasa stdev", error_y="pKa stdev",
#                 hover_data=["PDB Code", "resid", "chain", "Uniprot Entry"])
#fig.show()

In [20]:
import plotly.graph_objects as go
from ipywidgets import HBox, VBox

import numpy as np
np.random.seed(1)

x = np.random.rand(100)
y = np.random.rand(100)

e = go.FigureWidget([go.Scatter(x=x, y=y, mode='markers')])#, go.Scatter(x=x, y=y, mode='markers')])
g = go.FigureWidget([go.Scatter(x=x, y=y, mode='markers')])#, go.Scatter(x=x, y=y, mode='markers')])

scattere = e.data[0]
colors = ['#a3a7e4'] * 100
scattere.marker.color = colors
scattere.marker.size = [10] * 100
e.layout.hovermode = 'closest'

scatterg = g.data[0]
colors = ['#a3a7e4'] * 100
scatterg.marker.color = colors
scatterg.marker.size = [10] * 100
g.layout.hovermode = 'closest'

# create our callback function
def update_pointe(trace, points, selector):
    c = list(scattere.marker.color)
    s = list(scattere.marker.size)
    for i in points.point_inds:
        c[i] = '#bae2be'
        s[i] = 20
        with e.batch_update():
            scattere.marker.color = c
            scattere.marker.size = s
            
# create our callback function
def update_pointg(trace, points, selector):
    c = list(scatterg.marker.color)
    s = list(scatterg.marker.size)
    for i in points.point_inds:
        c[i] = '#bae2be'
        s[i] = 20
        with g.batch_update():
            scatterg.marker.color = c
            scatterg.marker.size = s



scattere.on_click(update_pointg)
scatterg.on_click(update_pointe)


VBox([e,g])

    'data': [{'marker': {'color': [#a3a7e4, #a3a7e4, #a3a7e4, #a3a7e4, #a3a7e4,
…

In [21]:
def set_opacity(opacity, size):
    scattere.marker.opacity = opacity
    scattere.marker.size = size

from ipywidgets import interactive
opacity_slider = interactive(set_opacity,
                            opacity=(0.0, 1.0, 0.01),
                            size=(1,10,0.25))


opacity_slider.children[0].layout.width = '400px'
opacity_slider.children[1].layout.width = '400px'

VBox([e,g, opacity_slider])

    'data': [{'marker': {'color': [#a3a7e4, #a3a7e4, #a3a7e4, #a3a7e4, #a3a7e4,
…